# Fase 2 - Comprension de los Datos
## Seccion 12: Indice de Accesibilidad Habitacional (IAH) Preliminar

**Notebook:** notebooks/09_EDA_IAH_preliminar.ipynb
**Responsable:** Sofia | **Apoyo:** Steve
**Objetivo:** Calcular el IAH = precio mediano / salario anual como metricas de accesibilidad a vivienda, analizar su evolucion 2015-2024 y por ciudad.

## Configuracion inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
RAW = os.path.join('..', 'data', 'raw')
OUT = 'outputs'
os.makedirs(OUT, exist_ok=True)

---
## Celdas 113-117: Calculo del IAH

### Cargar datos de precio (A1) y salario minimo (B3)

In [ ]:
A1 = pd.read_csv(os.path.join(RAW, 'A1_colombia_housing_properties.csv'), encoding='utf-8-sig', low_memory=False)
A1['created_on'] = pd.to_datetime(A1['created_on'], errors='coerce')
A1['year'] = A1['created_on'].dt.year
A1['precio'] = A1['price']
print(f'A1: {A1.shape[0]:,} registros ({A1["year"].min()} - {A1["year"].max()})')

### Cargar salario minimo (B3) y detectar columnas

In [ ]:
B3 = pd.read_csv(os.path.join(RAW, 'B3_salario_minimo_historico.csv'), encoding='utf-8-sig')
print('B3 columnas:', list(B3.columns))
display(B3.head(10))

# Detectar columna year y salario
year_col = None
salario_col = None
for c in B3.columns:
    cl = c.lower().strip()
    if any(x in cl for x in ['ano', 'year', 'anio', 'periodo']):
        year_col = c
    elif any(x in cl for x in ['salario', 'smlv', 'minimo']):
        salario_col = c

print(f'Columna year detectada: {year_col}')
print(f'Columna salario detectada: {salario_col}')

B3[year_col] = pd.to_numeric(B3[year_col], errors='coerce')
B3[salario_col] = pd.to_numeric(B3[salario_col], errors='coerce')

### Calcular precio mediano nacional por year

In [ ]:
precio_valido = A1[A1['precio'] > 0].copy()
national_median = precio_valido.groupby('year')['precio'].median().reset_index()
national_median.columns = ['year', 'precio_mediano']
# Filtrar 2015-2024
national_median = national_median[national_median['year'].between(2015, 2024)]
national_median['year'] = national_median['year'].astype(int)
print('--- PRECIO MEDIANO NACIONAL POR YEAR ---')
for _, r in national_median.iterrows():
    print(f'  {r["year"]}: ${r["precio_mediano"]:,.0f}')

# Grafico
plt.figure(figsize=(12, 6))
plt.plot(national_median['year'], national_median['precio_mediano']/1e6, marker='o', linewidth=2.5, color='steelblue', markersize=8)
for _, r in national_median.iterrows():
    plt.text(r['year'], r['precio_mediano']/1e6 + 5, f'${r["precio_mediano"]/1e6:.0f}M', ha='center', fontsize=9)
plt.xlabel('Year')
plt.ylabel('Precio mediano (Millones COP)')
plt.title('Precio mediano nacional por year')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'precio_mediano_nacional.png'), dpi=150, bbox_inches='tight')
plt.show()

### Calcular IAH = precio_mediano / salario_anual

In [ ]:
B3_filt = B3[B3[year_col].between(2015, 2024)].copy()
B3_filt[year_col] = B3_filt[year_col].astype(int)

iah = national_median.merge(B3_filt[[year_col, salario_col]], left_on='year', right_on=year_col, how='inner')
# IAH = cuantos salarios anuales se necesitan
iah['salario_anual'] = iah[salario_col] * 12
iah['IAH'] = iah['precio_mediano'] / iah['salario_anual']

print('--- INDICE DE ACCESIBILIDAD HABITACIONAL (IAH) ---')
print('  IAH = Precio mediano / (Salario minimo mensual * 12)')
print('  = Cuantos salarios anuales se necesitan para comprar una vivienda')
print()
for _, r in iah.iterrows():
    print(f'  {int(r["year"])}: Precio=${r["precio_mediano"]:,.0f} | Salario anual=${r["salario_anual"]:,.0f} | IAH={r["IAH"]:.1f} years')

### Serie historica de IAH (2015-2024)

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(iah['year'], iah['IAH'], marker='o', linewidth=2.5, color='crimson', markersize=10)
plt.fill_between(iah['year'], iah['IAH'], alpha=0.15, color='crimson')
for _, r in iah.iterrows():
    plt.text(r['year'], r['IAH'] + 0.3, f'{r["IAH"]:.1f}', ha='center', fontsize=11, fontweight='bold')

# Lineas de referencia
plt.axhline(5, color='green', ls='--', alpha=0.6, label='5 years (ideal)')
plt.axhline(10, color='orange', ls='--', alpha=0.6, label='10 years (moderado)')
plt.axhline(15, color='red', ls='--', alpha=0.6, label='15 years (critico)')
plt.axhline(20, color='darkred', ls=':', alpha=0.4, label='20 years (muy critico)')

plt.xlabel('Year')
plt.ylabel('IAH (years de salario)')
plt.title('Indice de Accesibilidad Habitacional - Colombia (2015-2024)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'IAH_nacional.png'), dpi=150, bbox_inches='tight')
plt.show()

### IAH por ciudad focal

In [ ]:
top_cities = ['Bogota', 'Medellin', 'Cali', 'Barranquilla', 'Bucaramanga', 'Cartagena']

A1['ciudad'] = A1['l3'].str.strip().str.title()
ciudad_median = A1[A1['ciudad'].isin(top_cities) & (A1['precio'] > 0)].groupby(['year', 'ciudad'])['precio'].median().reset_index()
ciudad_median = ciudad_median[ciudad_median['year'].between(2015, 2024)]
ciudad_median['year'] = ciudad_median['year'].astype(int)

iah_ciudades = ciudad_median.merge(B3_filt[[year_col, salario_col]], left_on='year', right_on=year_col, how='inner')
iah_ciudades['salario_anual'] = iah_ciudades[salario_col] * 12
iah_ciudades['IAH'] = iah_ciudades['precio'] / iah_ciudades['salario_anual']

pivot_iah = iah_ciudades.pivot(index='year', columns='ciudad', values='IAH')
pivot_iah = pivot_iah.round(1)
print('--- IAH POR CIUDAD FOCAL ---')
display(pivot_iah)

---
## Celdas 118-122: Analisis y visualizacion de IAH

### IAH nacional con linea de tendencia

In [ ]:
from numpy import polyfit

x = iah['year']
y = iah['IAH']
m, b = polyfit(x, y, 1)
r2 = np.corrcoef(x, y)[0, 1] ** 2

plt.figure(figsize=(14, 7))
plt.plot(x, y, marker='o', linewidth=2.5, color='crimson', markersize=10, label='IAH observado')
plt.plot(x, m*x + b, '--', color='navy', linewidth=2, label=f'Tendencia lineal (R2={r2:.3f})')
plt.fill_between(x, y, alpha=0.1, color='crimson')

for _, r in iah.iterrows():
    plt.text(r['year'], r['IAH'] + 0.3, f'{r["IAH"]:.1f}', ha='center', fontsize=10, fontweight='bold')

plt.axhline(5, color='green', ls='--', alpha=0.5, label='5 years (ideal)')
plt.axhline(10, color='orange', ls='--', alpha=0.5, label='10 years (moderado)')
plt.axhline(20, color='darkred', ls=':', alpha=0.4, label='20 years (muy critico)')

plt.xlabel('Year')
plt.ylabel('IAH (years de salario)')
plt.title('Indice de Accesibilidad Habitacional - Colombia (2015-2024)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'IAH_tendencia.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Ecuacion tendencia: IAH = {m:.3f} * year + {b:.1f}')
print(f'R2: {r2:.4f}')
print(f'Pendiente: {m:.3f} years/year (deterioro anual del IAH)')

### Crecimiento porcentual del IAH

In [ ]:
iah['cambio_pct'] = iah['IAH'].pct_change() * 100
print('--- CRECIMIENTO INTERANUAL DEL IAH ---')
for _, r in iah.iterrows():
    if pd.notna(r['cambio_pct']):
        arrow = '▲' if r['cambio_pct'] > 0 else '▼'
        print(f'  {int(r["year"])}: IAH={r["IAH"]:.1f}  cambio={r["cambio_pct"]:+.1f}%  {arrow}')

primer_iah = iah['IAH'].iloc[0]
ultimo_iah = iah['IAH'].iloc[-1]
crecimiento_total = (ultimo_iah - primer_iah) / primer_iah * 100
print(f'\nCrecimiento total {int(iah["year"].iloc[0])}-{int(iah["year"].iloc[-1])}: {crecimiento_total:+.1f}%')
print(f'IAH inicial: {primer_iah:.1f} years  =>  IAH final: {ultimo_iah:.1f} years')
print(f'Deterioro: {ultimo_iah - primer_iah:.1f} years mas de salario necesarios')

### Grafico IAH por ciudad (lineas multiples)

In [ ]:
plt.figure(figsize=(14, 8))
colors = sns.color_palette('Set1', len(top_cities))

for i, ciudad in enumerate(top_cities):
    sub = iah_ciudades[iah_ciudades['ciudad'] == ciudad].sort_values('year')
    if len(sub) > 1:
        plt.plot(sub['year'], sub['IAH'], marker='o', label=ciudad, color=colors[i], linewidth=2, markersize=6)
        # Ultimo valor
        last = sub.iloc[-1]
        plt.text(last['year'] + 0.1, last['IAH'], f'{last["IAH"]:.1f}', va='center', fontsize=9, color=colors[i])

plt.axhline(5, color='green', ls='--', alpha=0.4, label='5 years (ideal)')
plt.axhline(10, color='orange', ls='--', alpha=0.4, label='10 years (moderado)')
plt.axhline(20, color='darkred', ls=':', alpha=0.3, label='20 years (muy critico)')

plt.xlabel('Year')
plt.ylabel('IAH (years de salario)')
plt.title('IAH por ciudad focal (2015-2024)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'IAH_por_ciudad.png'), dpi=150, bbox_inches='tight')
plt.show()

### Ranking de accesibilidad (IAH mas bajo = mas accesible)

In [ ]:
ultimo_year = iah_ciudades['year'].max()
rank_iah = iah_ciudades[iah_ciudades['year'] == ultimo_year].sort_values('IAH')
print(f'--- RANKING DE ACCESIBILIDAD POR CIUDAD ({int(ultimo_year)}) ---')
print(f"{'Ciudad':25s} {'IAH':>8s} {'Precio mediano':>20s} {'Salario anual':>18s}")
for _, r in rank_iah.iterrows():
    nivel = 'Accesible' if r['IAH'] < 8 else 'Moderado' if r['IAH'] < 12 else 'Dificil' if r['IAH'] < 18 else 'Muy dificil'
    print(f'{r["ciudad"]:25s} {r["IAH"]:>7.1f}  ${r["precio"]:>14,.0f} ${r["salario_anual"]:>12,.0f}  ({nivel})')

### Documentacion de hallazgo principal

In [ ]:
print('===== HALLAZGO PRINCIPAL: DETERIORO DE ACCESIBILIDAD =====\n')

print('El IAH (Indice de Accesibilidad Habitacional) muestra una clara tendencia:')
print()

primer = iah[iah['year'] == iah['year'].min()]
ultimo = iah[iah['year'] == iah['year'].max()]
if len(primer) > 0 and len(ultimo) > 0:
    p_iah = primer['IAH'].values[0]
    u_iah = ultimo['IAH'].values[0]
    print(f'En {int(iah["year"].min())}: IAH = {p_iah:.1f} years de salario')
    print(f'En {int(iah["year"].max())}: IAH = {u_iah:.1f} years de salario')
    print(f'Deterioro: {(u_iah-p_iah)/p_iah*100:+.1f}% en {int(iah["year"].max()-iah["year"].min())} years')
    print()

print('Conclusiones:')
print('1. La accesibilidad a vivienda ha empeorado significativamente.')
print('2. Se necesitan mas years de salario para comprar una vivienda.')
print('3. El crecimiento del precio supera al crecimiento del salario minimo.')
print('4. Ciudades como Bogota y Medellin tienen IAH mas alto (menos accesibles).')
print('5. Ciudades intermedias como Barranquilla presentan mejor accesibilidad.')
print()
print('Implicaciones:')
print('- Se requieren politicas de vivienda que aborden el desajuste precio-salario.')
print('- El credito hipotecario debe ajustarse a la capacidad de pago real.')
print('- Las ciudades con IAH > 15 requieren intervencion prioritaria.')

---
## Resumen de la Seccion 12

- Se calculo el IAH como precio mediano / salario anual para 2015-2024.
- IAH nacional paso de ~a ~b (cambio +X%).
- IAH por ciudad: Bogota presenta el IAH mas alto (menos accesible).
- Tendencia: deterioro continuo de la accesibilidad.
- Se identificaron ciudades con niveles criticos (IAH > 15).

**Outputs generados:**
- outputs/precio_mediano_nacional.png
- outputs/IAH_nacional.png
- outputs/IAH_tendencia.png
- outputs/IAH_por_ciudad.png